# Book-Crossing Preprocessing

This notebook is organized for the graph-based book recommendation project.
It loads the raw CSV files, cleans the interactions, plots the requested charts, and saves the final dataset to `data/processed/cleaned_ratings.csv`.

In [1]:
from pathlib import Path
import sys

import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.book_reco.preprocessing import (
    build_summary_statistics,
    drop_duplicate_rows,
    filter_explicit_ratings,
    load_raw_datasets,
    plot_preprocessing_visualizations,
    prepare_cleaned_ratings,
    report_dataset_overview,
    report_missing_values,
    save_cleaned_ratings,
)

In [2]:
# Load the raw Book-Crossing tables from the repository data folder.
books, ratings, users = load_raw_datasets()
datasets = {"Books": books, "Ratings": ratings, "Users": users}

report_dataset_overview(datasets)
report_missing_values(datasets)

FileNotFoundError: [Errno 2] No such file or directory: 'data\\raw-dataset-books\\Books.csv'

In [ ]:
# Remove duplicate rows before any modeling or graph construction.
books, ratings, users = drop_duplicate_rows(books, ratings, users)
print("\nShapes after removing duplicate rows:")
for name, dataframe in {"Books": books, "Ratings": ratings, "Users": users}.items():
    print(f"{name}: {dataframe.shape}")

# Keep only explicit ratings, because ratings equal to zero are implicit feedback.
ratings = filter_explicit_ratings(ratings)
print("\nRatings shape after keeping explicit ratings only:", ratings.shape)
print("Unique ratings remaining:", ratings["Book-Rating"].value_counts().sort_index().to_dict())

# Merge the ratings with book metadata and remove sparse users/books until stable.
cleaned_ratings = prepare_cleaned_ratings(books, ratings)
print("Shape after removing sparse users and books:", cleaned_ratings.shape)

In [ ]:
# Create the summary statistics requested for the cleaned interaction table.
summary_stats = build_summary_statistics(cleaned_ratings)
print(summary_stats.to_string(index=False))

In [ ]:
# Visualize the rating distribution, the most-rated books, and the most-active users.
plot_preprocessing_visualizations(cleaned_ratings)

In [ ]:
# Save the cleaned dataset in the processed data folder for downstream GNN work.
output_path = save_cleaned_ratings(cleaned_ratings)
print(f"Cleaned dataset saved to: {output_path.resolve()}")

# Show a compact preview of the final table.
cleaned_ratings.head()

## LightGCN Data Reduction

The EDA showed that the full Book-Crossing interaction graph is extremely sparse and dominated by implicit zero ratings. For LightGCN with BPR ranking, retained edges should represent positive user-book preference signals rather than ambiguous non-ratings.

Reduction policy:

- Remove duplicate and invalid interaction records.
- Keep only ratings whose users and ISBNs still exist in the cleaned user/book tables.
- Treat `Book-Rating = 0` as implicit exposure or missing preference, not as a positive edge.
- Keep explicit positive interactions with `Book-Rating >= 5`.
- Compare candidate k-core thresholds before selecting the final graph.
- Use an iterative `user >= 10` and `book >= 10` k-core for the final LightGCN dataset.

The EDA suggested trying `book >= 20`, but the candidate analysis below shows that after removing implicit zero ratings this threshold removes the entire explicit-preference graph. The final `book >= 10` threshold is therefore used because it preserves a trainable graph while still requiring each retained book to have enough evidence for collaborative filtering.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.reduce_book_crossing_for_lightgcn import main as reduce_for_lightgcn

# Rebuild the reduced graph-ready artifacts from the EDA-cleaned tables.
reduce_for_lightgcn()

processed_dir = REPO_ROOT / "data" / "processed"

reduction_stats = pd.read_csv(processed_dir / "data_reduction_statistics.csv")
candidate_stats = pd.read_csv(processed_dir / "threshold_candidate_analysis.csv")
graph_stats = pd.read_csv(processed_dir / "graph_statistics_reduced_lightgcn.csv")

display(reduction_stats)
display(candidate_stats)
display(graph_stats)

print("Final LightGCN-ready files:")
for filename in [
    "ratings_reduced_lightgcn.csv",
    "books_reduced_lightgcn.csv",
    "users_reduced_lightgcn.csv",
    "merged_reduced_lightgcn.csv",
    "cleaned_ratings.csv",
    "user_mapping.csv",
    "book_mapping.csv",
    "edge_index.csv",
    "edge_weight.csv",
    "lightgcn_graph.pt",
    "artifact_manifest_reduced_lightgcn.csv",
]:
    path = processed_dir / filename
    print(f"- {filename}: {path.exists()} ({path.stat().st_size if path.exists() else 0:,} bytes)")
